# CancerFusion: leakage-restricted survival study

This notebook runs the corrected shared experiment engine, not the legacy pilot head. **Validated means engineering checks, not clinical validation.** Labels are recorded OS/time/censoring; no outcome-generated classification labels are used. Age is the strict diagnosis-time clinical baseline. Pathologic-stage analyses are exploratory because the recorded OS origin and staging-completion landmark are not guaranteed aligned. Earlier legacy runs used overlapping TCGA patients, so the new held-out split is not a globally untouched external cohort.

Attach the private dataset produced by `python -m tools.build_kaggle_study_bundle`. It contains a checksummed `cancerfusion-study-code/` tree, processed patient inputs and checksum manifest. Network access is unnecessary once inputs are attached. This engine uses CPU full-batch training for its approximately1080patient cohort; allocating a GPU does not automatically change its device. No accuracy or C-index is promised. Raw image encoders and four-modality claims require real verified image inputs and are not fabricated here.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile, platform, shutil
import numpy as np, pandas as pd, scipy, sklearn, torch, openpyxl
print({'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'openpyxl':openpyxl.__version__})
manifests = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
if len(manifests) != 1:
    raise ValueError(f'Attach exactly one prepared CancerFusion dataset; found {len(manifests)} manifests.')
INPUT = manifests[0].parent
manifest = json.loads((INPUT/'bundle_manifest.json').read_text())
for relative, expected in manifest['sha256'].items():
    path = (INPUT/relative).resolve()
    if not path.is_relative_to(INPUT.resolve()) or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Input checksum/path failure: {relative}')
WORK = Path('/kaggle/working/cancerfusion-study')
WORK.mkdir(parents=True, exist_ok=True)
CODE = INPUT/'cancerfusion-study-code'
if not CODE.is_dir():
    raise ValueError(f'Expected extracted, checksummed code directory: {CODE}')
shutil.copytree(CODE, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0,str(WORK))
DATA = INPUT/'study-data'
# Preserve the original relative source layout required by the locked protocol
# and the separately audited author-model runner. Files remain byte-identical.
for relative in manifest.get('project_layout_inputs',[]):
    source = DATA/relative
    target = WORK/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,target)
print('Verified private input bundle:',manifest['built_at_utc'])


## Configuration and engineering checks

Three training seeds42/123/2026; training-only preprocessing and gene selection. BRCA uses its existing60/20/20 patient split. The separate KIRC study uses the locked five outer folds and fixed inner validation assignments described below. Event counts, available inputs and unsupported horizons are exported. Conditional patient-bootstrap intervals do not include retraining uncertainty or overlap among development folds. Library versions are recorded; identical input/code hashes do not guarantee bitwise equality on a different runtime. Do not choose the best test configuration and claim superiority.


In [ ]:
import subprocess
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EPOCHS = 300
RNA_GENES = 256
RUN_CLINICAL = True
RUN_CLI_RNA = True
RUN_EXTERNAL_RANK_TRANSPORT = True
RUN_KIRC_CLI_RNA = 'TCGA-KIRC' in manifest['cohorts']
RUN_KIRC_AUTHOR_MULTISURV_RNA = RUN_KIRC_CLI_RNA
# Image acquisition alone is insufficient: a reviewed QC-cohort lock and
# verified feature matrices are required before any image efficacy experiment.
RUN_IMAGE_FOUR_INPUT_STUDY = False
OUT = WORK/'RUN REPORT'
OUT.mkdir(exist_ok=True)
def run_module(module, *arguments):
    command = [sys.executable,'-m',module,*map(str,arguments)]
    print('Running:',command)
    subprocess.run(command,check=True)
run_module('tools.test_survival_models')
run_module('tools.test_survival_metrics')
if RUN_KIRC_AUTHOR_MULTISURV_RNA:
    run_module('tools.test_published_baselines')


## A. Recorded-outcome clinical baselines

Age-only Cox is the strict shared baseline. Age+pathologic-stage Cox is an exploratory sensitivity analysis; source staging date versus OS origin must be reviewed before a clinical landmark claim. A conventional censored-hazard neural model and between-seed disagreement are evaluated, without claiming calibrated evidential uncertainty. METABRIC external predictions use a frozen age-only model; target outcomes estimate only the evaluation censoring nuisance distribution, not model parameters.


In [ ]:
if RUN_CLINICAL:
    run_module('tools.run_survival_experiments',
        '--input',DATA/'tcga_brca_clinical.csv',
        '--external',DATA/'metabric_confirmed_carcinoma_clinical.csv',
        '--epochs',EPOCHS,'--output',OUT/f'clinical-{RUN_ID}')


## B. Actual matched clinical+RNA experiments and ablations

This uses full downloaded and checksum-verified primary-tumour RNA inputs, not outcomes alone. The clinical branch uses age only. Top256variable genes are selected on training patients only. Compare unimodal, concatenation, mean and attention fusion; no-mask and training-dropout0.1/0.3/0.5; separately retrained leave-one-input-out models; and inference withholding on identical complete-input held-out cases. With only two actual modalities these are **two-modality experiments**, not three/four-modality results. No WSI or radiology performance is inferred.


In [ ]:
if RUN_CLI_RNA:
    run_module('tools.run_survival_experiments',
        '--input',DATA/'tcga_brca_clinical.csv','--clinical-columns','age_years',
        '--rna-npz',DATA/'tcga_brca_rna_logtpm.npz','--rna-genes',RNA_GENES,
        '--complete-cohort','--epochs',EPOCHS,'--output',OUT/f'cli-rna-{RUN_ID}')


## C. Separate exploratory cross-assay external transfer

Raw METABRIC microarray values cannot pass through an RNA-seq logTPM scaler. This is a **different representation and different trained model**: source/target shared gene symbols are fixed, duplicate symbols averaged within patient, patient-level ranks computed, then gene selection/scaling fitted only on TCGA training. The model is frozen before METABRIC outcomes are evaluated. This is an exploratory portability study, not proof that rank normalization removes domain shift.


In [ ]:
if RUN_EXTERNAL_RANK_TRANSPORT:
    run_module('tools.run_survival_transport_experiment',
        '--source-clinical',DATA/'tcga_brca_clinical.csv',
        '--source-rna',DATA/'tcga_brca_rna_logtpm.npz',
        '--target-clinical',DATA/'metabric_confirmed_carcinoma_clinical.csv',
        '--target-rna',DATA/'metabric_microarray.npz',
        '--genes',RNA_GENES,'--epochs',EPOCHS,'--output',OUT/f'rank-transfer-{RUN_ID}')


## D. Separate retrospective KIRC clinical + RNA study: five locked folds

The `--include-kirc` bundle contains217 patients/66 recorded OS deaths, the original CDR workbook, protocol and all five immutable patient ledgers. Original OS/time are retained. RNA procurement timing is unknown; this is retrospective association, without a prospective diagnosis-time claim. GDC `days_to_collection` is center receipt and is never substituted as tissue procurement or a landmark. No receipt-date exclusions are applied. Kidney and breast cohorts remain separate.

Each fold compares matched age-only Cox, clinical/RNA unimodal networks, concatenation, mean and attention fusion, masking/dropout0/0.1/0.3/0.5, retrained leave-one-input-out models and inference withholding. Top256variable genes and scalers are fitted only on that fold's training patients. The locked primary estimator is the equal mean of within-fold C, with paired patient resampling **within** fixed folds. Secondary pooled OOF scores use negative mean predicted survival at365/1095/1825days for every model, including Cox; unscaled Cox linear predictors are never pooled. The two-input three-subset experiment does not test the planned four-input15-subset hypothesis.


In [ ]:
if RUN_KIRC_CLI_RNA:
    # This fixed relative output identifier is also the frozen reference used
    # by the separately executed author-model driver. A rerun never overwrites it.
    KIRC_OUT = OUT/'kirc-cli-rna-20261118-fixed5fold-v1'
    KIRC_OUT.mkdir(exist_ok=False)
    for fold in range(1,6):
        run = KIRC_OUT/f'fold_{fold:02d}'
        run_module('tools.run_survival_experiments',
            '--input',WORK/'data/processed/tcga_kirc_clinical.csv',
            '--clinical-columns','age_years',
            '--rna-npz',WORK/'data/processed/tcga_kirc_rna_logtpm.npz',
            '--rna-genes',256,'--complete-cohort',
            '--split-csv',WORK/f'data/processed/kirc-fixed-folds/fold_{fold:02d}.csv',
            '--study-protocol',WORK/'config/kirc_retrospective_os_protocol.json',
            '--bootstrap',1000,'--bootstrap-seed',20261118,
            '--epochs',300,'--output',run)
        run_module('tools.independently_validate_survival_run','--run',run)
    run_module('tools.summarize_survival_folds','--runs-root',KIRC_OUT)
    run_module('tools.independently_validate_survival_folds','--root',KIRC_OUT)
    run_module('tools.plot_survival_fold_summary','--runs-root',KIRC_OUT)
else:
    print('Separate KIRC clinical/RNA study skipped: optional verified inputs were not bundled.')


## E. Separate adapted published baseline: author MultiSurv mRNA network/loss

This executes a separate RNA-only baseline from the pinned author repository commit `deb35e12502f61a59726bfb543210735406d5959`; original MIT source/license are included and checked. It retains the unmodified author mRNA network,30annual conditional-survival outputs and original midpoint-censor likelihood. The217-case KIRC cohort, locked folds, log1pTPM top1000training-variance genes/scaling, Adam0.0001 and early-stopping policy are explicit shared-cohort adaptations. This is not the original pan-cancer study, nine-category clinical pathway, ResNeXt WSI pathway, CT model or a four-input MultiSurv reproduction. The benchmark was added after the initial CLI/RNA analysis and is secondary. Different gene budgets and training policies prevent a clean architectural-superiority claim.

The author-model driver and validator use the original project-relative paths and therefore remain unmodified. Run the KIRC reference step first. The independent audit reconstructs checkpoints, preprocessing, likelihood, original CDR outcomes, predictions and paired fold summaries. Original float32 CSV rendering can round decimal text; the validator writes a separate full-precision reconstruction and preserves the original artifact/hash.


In [ ]:
if RUN_KIRC_AUTHOR_MULTISURV_RNA:
    if not RUN_KIRC_CLI_RNA or not (OUT/'kirc-cli-rna-20261118-fixed5fold-v1/five_fold_summary.json').is_file():
        raise ValueError('Run and validate the separate locked KIRC reference study first.')
    run_module('tools.run_author_multisurv_rna')
    run_module('tools.validate_author_multisurv_rna')
else:
    print('Adapted author MultiSurv RNA study skipped: KIRC reference inputs are absent.')


## Interpretation and uncertainty: preserve the negative findings

The independently audited local KIRC study did not establish multimodal superiority: all paired improvement intervals versus age-only Cox crossed zero. For the exploratory two-input missingness panel, dropout0.3 minus0 was0.00789,95% conditional CI[−0.00396,0.02013]. Attention's5-year error/disagreement correlations varied approximately−0.299 to+0.131 across folds and its selective-risk curves were inconsistent. These are three-seed disagreement scores; calibrated confidence and evidential-survival uncertainty remain unestablished. The planned four-input hypothesis remains untested.

Recompute findings from this run; these local observations are not acceptance targets or promised accuracy. Images/four-input execution stays OFF until a reviewed QC-cohort lock and verified feature matrices are available. Source availability and encoder extraction do not establish tumour coverage, diagnosis-time eligibility or clinical validation.


## Results, figures and downloadable artifacts

Read results from exported predictions and metric files. Clinical classification accuracy is not an appropriate replacement for censored survival metrics. Intervals crossing zero in paired differences do not establish improvement. Ensemble disagreement/error association is descriptive and may fail to identify poor predictions. All data-constrained or unsupported metrics must remain explicit.


In [ ]:
rows=[]
for folder in sorted(OUT.iterdir()):
    if (folder/'five_fold_summary.json').is_file():
        combined=json.loads((folder/'five_fold_summary.json').read_text())
        for name,value in combined['primary_locked_within_fold_summary'].items():
            rows.append({'run':folder.name,'model':name,'scope':'primary_mean_within_fold_c','patients':combined['oof_patients'],'events':combined['oof_observed_deaths'],'harrell_c':value['mean_within_fold_c'],'bootstrap_95_ci':value['conditional_fold_stratified_patient_bootstrap_95_ci']})
        for name,value in combined['secondary_pooled_oof_common_probability_summary'].items():
            rows.append({'run':folder.name,'model':name,'scope':'secondary_pooled_oof_common_probability_c','patients':combined['oof_patients'],'events':combined['oof_observed_deaths'],'harrell_c':value['harrell_c'],'bootstrap_95_ci':value['bootstrap_95_ci']})
        continue
    if folder.name=='kirc-multisurv-author-rna-20261118-v1':
        author=json.loads((folder/'independent_validation.json').read_text())['aggregate']
        rows.append({'run':folder.name,'model':'MultiSurv-author-mRNA shared-cohort adaptation','scope':'secondary_mean_within_fold_c','patients':author['patients'],'events':author['deaths'],'harrell_c':author['mean_outer_fold_c'],'bootstrap_95_ci':author['conditional_patient_bootstrap_95_ci']})
        continue
    if not (folder/'metrics.json').exists():continue
    measured=json.loads((folder/'metrics.json').read_text())
    for name,result in measured.items():
        for scope in ('test','internal','external'):
            value=result.get(scope)
            if isinstance(value,dict) and 'harrell_c' in value:
                rows.append({'run':folder.name,'model':name,'scope':scope,'patients':value.get('patients'),'events':value.get('observed_events'),'harrell_c':value['harrell_c'],'bootstrap_95_ci':value.get('bootstrap_95_ci')})
    if (folder/'test_predictions.csv').exists():
        run_module('tools.independently_validate_survival_run','--run',folder)
        if (folder/'inference_withholding_predictions.csv').exists():
            run_module('tools.audit_multimodal_robustness','--run',folder)
        run_module('tools.plot_survival_results','--run',folder)
    elif (folder/'predictions.csv').exists():
        run_module('tools.independently_validate_transport_run','--run',folder)
        run_module('tools.plot_transport_results','--run',folder)
pd.DataFrame(rows).to_csv(OUT/'all_run_results.csv',index=False)
display(pd.DataFrame(rows))
output_zip=Path('/kaggle/working/cancerfusion_validated_outputs.zip')
with zipfile.ZipFile(output_zip,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file():archive.write(path,path.relative_to(WORK))
print('Download:',output_zip)


Primary methods: [Kvamme and Borgan2021, censored discrete hazards](https://pmc.ncbi.nlm.nih.gov/articles/PMC8536596/); recorded endpoint source [TCGA-CDR](https://gdc.cancer.gov/about-data/publications/PanCan-Clinical-2018); external source [cBioPortal METABRIC](https://github.com/cBioPortal/datahub/tree/master/public/brca_metabric); [pinned author MultiSurv source](https://github.com/luisvalesilva/multisurv/tree/deb35e12502f61a59726bfb543210735406d5959). The explicit mRNA-only author adaptation above does not reproduce the full MultiSurv paper. MCAT/PathomicFusion, image-cohort efficacy and evidential-survival uncertainty remain separate requirements.
